# S-DiverSe × Whisper: reconocimiento de habla de personas con ELA, Parkinson y ACV

Este notebook compara cuatro enfoques sobre **S-DiverSe** (3,2 h de habla espontánea en español, 22 hablantes,
444 segmentos; López et al., Interspeech 2026, arXiv:2607.03207):

| Enfoque | Qué es |
|---|---|
| **A** | `openai/whisper-small` sin entrenar |
| **A+PP** | A + las 3 reglas de post-procesamiento del paper (borran repeticiones alucinadas) |
| **B** | A+PP + corrección con un LLM que ve varias hipótesis del beam search |
| **C** | whisper-small con fine-tuning LoRA sobre hablantes de entrenamiento |
| **C+PP** | C + las reglas |
| **D** | C+PP + corrección con LLM |
| Ref. | `whisper-large-v3-turbo` sin entrenar (para responder "¿y contra un Whisper grande?") |

**Regla de oro:** test separado **por hablante**. Ningún hablante de test aparece en entrenamiento ni en validación.

Funciona en **Google Colab** y en **Kaggle** (lo detecta solo). Correlo de punta a punta: cada etapa guarda sus
resultados y, si la sesión se corta, al volver a correr retoma desde donde quedó.

## 1. Setup

**Dependencias.** Fijo versiones de las librerías que cambian su API seguido (`transformers`, `peft`, `jiwer`), para
que el notebook se comporte igual que cuando lo probé. `huggingface-hub<1.0` es obligatorio: sin ese límite, instalar
`gradio` actualiza `huggingface-hub` a 2.x y rompe `transformers` 4.57. `num2words` pasa números a palabras para el WER; `anthropic` y
`openai` son los dos clientes de LLM posibles (enfoques B y D); `gradio` es para la demo.

In [ ]:
%pip install -q "transformers==4.57.1" "peft==0.17.1" "accelerate>=1.0" "jiwer==4.0.0" num2words anthropic openai "gradio>=4.44" "huggingface-hub>=0.34,<1.0"

**Configuración.** Todo lo que se puede tocar está en `CFG`. Explicación de cada hiperparámetro de LoRA en la sección 7.

- En **Kaggle** elegí acelerador *GPU T4 x2*; uso una sola GPU (`CUDA_VISIBLE_DEVICES=0`) porque el entrenamiento
  multi-GPU con PEFT agrega problemas y no hace falta para un modelo de este tamaño.
- `llm_proveedor`: por defecto **Groq** (`"openai_compatible"` con la URL de Groq). También `"anthropic"` (SDK
  oficial) o `"ninguno"` (salta B y D).
- La API key se lee de: variable de entorno → *Secrets* de Colab (ícono de llave) → *Add-ons → Secrets* de Kaggle.
  Nombres: `ANTHROPIC_API_KEY`, `GROQ_API_KEY` (si `llm_base_url` es de Groq) u `OPENAI_API_KEY`.
- **Groq** (gratis) usa 2 hilos porque el plan gratuito limita las llamadas por minuto; si una llamada choca con el
  límite, el notebook espera y reintenta.

In [ ]:
import os, sys, json, re, math, time, random, shutil, zipfile, unicodedata, glob, hashlib
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np
import pandas as pd
import torch

EN_KAGGLE = os.path.exists("/kaggle/working")
EN_COLAB = (not EN_KAGGLE) and ("google.colab" in sys.modules or os.path.exists("/content"))

CFG = dict(
    modelo_base="openai/whisper-small",
    modelo_referencia="openai/whisper-large-v3-turbo",
    evaluar_referencia=True,          # Whisper grande sin entrenar sobre val+test (~5 min)
    alineador="jonatasgrosman/wav2vec2-large-xlsr-53-spanish",
    seed=42,
    # --- LoRA (explicado en la sección 7) ---
    lora_r=16, lora_alpha=32, lora_dropout=0.1,
    lr=5e-4, epocas=8, batch=8, grad_accum=2, warmup_ratio=0.1, paciencia=3,
    max_trozo_s=28.0,                 # largo máximo de cada trozo de entrenamiento (Whisper ve 30 s)
    # --- decodificación ---
    nbest=5,                          # hipótesis del beam search que ve el LLM (solo segmentos <= 30 s)
    # --- LLM (enfoques B y D) ---
    llm_proveedor="openai_compatible",  # Groq (API compatible con OpenAI). Otras: "anthropic" | "ninguno"
    llm_modelo="openai/gpt-oss-120b",   # el que usamos; disponibles: console.groq.com/docs/models
    llm_esfuerzo="low",               # solo Anthropic: esfuerzo de razonamiento. None para no enviarlo
    llm_base_url="https://api.groq.com/openai/v1",
    llm_hilos=2,                      # pocas llamadas en paralelo: el plan gratis de Groq limita por minuto
    llm_ejemplos=3,                   # ejemplos (few-shot) tomados de hablantes de ENTRENAMIENTO
    # --- validación cruzada opcional ---
    ejecutar_cv=False,
)
# Por defecto usa Groq con el secreto GROQ_API_KEY. Para usar Claude en su lugar, descomentá esta línea
# y cargá el secreto ANTHROPIC_API_KEY:
# CFG.update(llm_proveedor="anthropic", llm_base_url=None, llm_modelo="claude-haiku-5-5", llm_hilos=8)

if EN_KAGGLE:
    BASE_DEF = "/kaggle/working/sdiverse_hackathon"
elif EN_COLAB:
    BASE_DEF = "/content/drive/MyDrive/Shippear/sdiverse_hackathon"
else:
    BASE_DEF = os.path.abspath("sdiverse_hackathon")
BASE = os.environ.get("SDIVERSE_BASE", BASE_DEF)        # carpeta PERSISTENTE (Drive o /kaggle/working)
DATOS = os.environ.get("SDIVERSE_DATOS", "/tmp/sdiverse_datos")  # copia local rápida de los audios
RES = os.path.join(BASE, "resultados")

def fijar_semilla(s=CFG["seed"]):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
fijar_semilla()

print("Entorno:", "Kaggle" if EN_KAGGLE else "Colab" if EN_COLAB else "local")
print("torch", torch.__version__)

**Montar Drive (solo Colab) y verificar GPU.** El modelo se usa en fp16 en GPU. Sin GPU el notebook corre igual
pero el fine-tuning se vuelve inviable (horas).

In [ ]:
if EN_COLAB and not os.environ.get("SDIVERSE_BASE"):
    from google.colab import drive
    drive.mount("/content/drive")
for d in (BASE, RES, DATOS):
    os.makedirs(d, exist_ok=True)
if EN_KAGGLE and not os.listdir(RES):
    # Reanudar en Kaggle: si agregaste como Input la salida de una corrida anterior, la copio.
    previas = glob.glob("/kaggle/input/**/sdiverse_hackathon", recursive=True)
    if previas:
        shutil.copytree(previas[0], BASE, dirs_exist_ok=True)
        print("Retomando desde la salida anterior:", previas[0])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory/1e9:.1f} GB")
    if "T4" not in p.name:
        print("Aviso: los tiempos estimados están calculados para una T4.")
else:
    print("SIN GPU. Colab: Entorno de ejecución → Cambiar tipo → T4. Kaggle: Settings → Accelerator → GPU T4 x2.")
print("Resultados persistentes en:", BASE)

## 2. Carga del zip

Busco `sdiverse_data.zip` en Drive (Colab) o en `/kaggle/input` (Kaggle; si subiste el zip como *Dataset*, Kaggle
ya lo descomprime y busco directamente el TSV). Lo descomprimo en el disco local de la máquina: leer 444 archivos
desde Drive es lento.

Después verifico cada audio contra el `manifest.csv` (hash SHA-1). Si algún archivo falta o está corrupto lo
informo y lo excluyo, en lugar de fallar más adelante.

In [ ]:
def buscar_datos():
    tsv = glob.glob(os.path.join(DATOS, "**", "s-diverse.tsv"), recursive=True)
    if tsv:
        return os.path.dirname(tsv[0])
    candidatos = []
    if EN_KAGGLE:
        candidatos += glob.glob("/kaggle/input/**/s-diverse.tsv", recursive=True)
        if candidatos:
            return os.path.dirname(candidatos[0])
        zips = glob.glob("/kaggle/input/**/sdiverse_data.zip", recursive=True)
    else:
        # Cualquier .zip en Mi unidad/Shippear/ (o en la carpeta de resultados) que contenga s-diverse.tsv
        zips = []
        for z in sorted(glob.glob("/content/drive/MyDrive/Shippear/*.zip") + glob.glob(os.path.join(BASE, "*.zip"))
                        + glob.glob(os.path.join(os.path.dirname(BASE), "*.zip"))):
            try:
                if any(n.endswith("s-diverse.tsv") for n in zipfile.ZipFile(z).namelist()):
                    zips.append(z)
            except zipfile.BadZipFile:
                print("No es un zip válido (¿subida incompleta?):", z)
    if not zips:
        raise FileNotFoundError("No encuentro el zip de datos. Colab: subilo a 'Mi unidad/Shippear/'. "
                                "Kaggle: subilo como Dataset y agregalo con '+ Add Input'.")
    print("Descomprimiendo", zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(DATOS)
    return os.path.dirname(glob.glob(os.path.join(DATOS, "**", "s-diverse.tsv"), recursive=True)[0])

RAIZ = buscar_datos()
df = pd.read_csv(os.path.join(RAIZ, "s-diverse.tsv"), sep="\t")
df["ruta"] = [os.path.join(RAIZ, "audio", d, f"{s}.wav") for d, s in zip(df.Disease, df.Sample_ID)]

man_path = os.path.join(RAIZ, "manifest.csv")
if os.path.exists(man_path):
    man = pd.read_csv(man_path).set_index("Sample_ID")
    def _ok(r):
        if not os.path.exists(r.ruta):
            return False
        with open(r.ruta, "rb") as f:
            return hashlib.sha1(f.read()).hexdigest() == man.loc[r.Sample_ID, "sha1"]
    df["audio_ok"] = df.apply(_ok, axis=1)
else:
    df["audio_ok"] = df.ruta.map(os.path.exists)
faltan = df[~df.audio_ok]
print(f"Segmentos: {len(df)} | audios OK: {df.audio_ok.sum()} | faltantes o corruptos: {len(faltan)}")
if len(faltan):
    print(faltan[["Sample_ID", "Speaker_ID"]].to_string())
df = df[df.audio_ok].reset_index(drop=True)

## 3. Exploración breve

Puntos a tener presentes al leer los resultados:
- **Inteligibilidad ≈ hablante.** 21 de 22 hablantes tienen un único nivel, y `High` son solo hablantes con Parkinson y
  `High/Medium` solo con ELA. El WER por inteligibilidad está mezclado con hablante y enfermedad.
- **Desbalance.** ELA es el 78 % de las horas; un solo hablante (ALS_M05) aporta ~49 min.
- **Duración.** 118 segmentos duran más de 30 s, que es lo máximo que Whisper procesa de una vez.

In [ ]:
import matplotlib.pyplot as plt

por_hablante = df.groupby("Speaker_ID").agg(
    enfermedad=("Disease", "first"), sexo=("Sex", "first"), segmentos=("Sample_ID", "size"),
    minutos=("Duration", lambda x: round(x.sum() / 60, 1)),
    inteligibilidad=("Intelligibility", lambda x: ", ".join(sorted(set(x))))).sort_values(["enfermedad", "segmentos"])
display(por_hablante)
display(pd.crosstab(df.Disease, df.Intelligibility, margins=True))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(df.Duration, bins=40, color="#4C72B0")
ax[0].axvline(30, color="crimson", ls="--", label="ventana de Whisper (30 s)")
ax[0].set(xlabel="duración del segmento (s)", ylabel="segmentos"); ax[0].legend()
por_hablante.segmentos.plot.bar(ax=ax[1], color=por_hablante.enfermedad.map(
    {"ALS": "#4C72B0", "PARKINSON": "#55A868", "STROKE": "#C44E52"}))
ax[1].set(ylabel="segmentos", xlabel=""); ax[1].tick_params(axis="x", labelsize=7)
plt.tight_layout(); plt.show()
print(f"Total: {df.Duration.sum()/3600:.2f} h | >30 s: {(df.Duration > 30).sum()} segmentos")

### Normalización del texto y cálculo del WER

Antes de comparar, referencia e hipótesis pasan por la misma `normalizar()`. Sigo al paper para que los números sean
comparables:
1. Unicode NFC y minúsculas.
2. Borro `<unk>` (marca de tramo ininteligible de los anotadores).
3. Números con dígitos → palabras en español (`2009` → `dos mil nueve`; `1.000` → `mil`). Whisper a veces escribe
   dígitos y a veces palabras: sin este paso, cada "20" contaría como error.
4. Quito toda puntuación (incluye `¿ ¡ « » " —`).
5. **Tildes:** en la métrica principal **se conservan**, porque en español distinguen palabras ("esta/está",
   "si/sí"). Reporto también un WER **sin tildes** como secundario. La ñ se conserva siempre.
6. Muletillas ("eh", "mmm") se mantienen, igual que en el paper.

**Agregación:** WER = (sustituciones + borrados + inserciones) / palabras de referencia, **sumando sobre todos los
segmentos del grupo** (no promedio de WER por segmento), igual que el paper. Así un segmento de 3 palabras no pesa lo
mismo que uno de 300. Cada tabla muestra cuántos segmentos, hablantes y palabras hay detrás de cada número.
Los segmentos cuya referencia queda vacía tras normalizar (solo `<unk>`) se excluyen del WER.

**Reglas del paper (PP).** Tres pasos en orden: (i) palabras de más de 15 caracteres hechas de una unidad repetida
se reducen a la unidad ("jajajaja..." → "ja"); (ii) palabras repetidas consecutivas se colapsan; (iii) frases
repetidas consecutivas se reducen a una. Atacan el error típico de Whisper con habla atípica: quedarse en bucle.

In [ ]:
import jiwer
from num2words import num2words

def _num_a_palabras(m):
    s = m.group(0).replace(".", "")
    try:
        return " " + num2words(int(s), lang="es") + " "
    except Exception:
        return " " + s + " "

def normalizar(texto, quitar_tildes=False):
    t = unicodedata.normalize("NFC", str(texto)).lower()
    t = t.replace("<unk>", " ")
    t = re.sub(r"\d{1,3}(?:\.\d{3})+(?!\d)|\d+", _num_a_palabras, t)
    t = re.sub(r"[^\w\s]|_", " ", t)
    if quitar_tildes:
        t = unicodedata.normalize("NFD", t)
        t = re.sub("[̀́̈]", "", t)   # grave, agudo, diéresis (la virgulilla de la ñ queda)
        t = unicodedata.normalize("NFC", t)
    return " ".join(t.split())

def contar_errores(ref, hyp):
    """Errores de un segmento con jiwer. ref/hyp ya normalizados."""
    r, h = ref.split(), hyp.split()
    if not r:
        return dict(S=0, D=0, I=len(h), N=0)
    if not h:
        return dict(S=0, D=len(r), I=0, N=len(r))
    o = jiwer.process_words(ref, hyp)
    return dict(S=o.substitutions, D=o.deletions, I=o.insertions, N=o.hits + o.substitutions + o.deletions)

def wer_segmentos(tabla, col_hyp, sin_tildes=False):
    """Una fila por segmento con S, D, I, N y WER."""
    filas = []
    for ref, hyp in zip(tabla.Transcription, tabla[col_hyp].fillna("")):
        filas.append(contar_errores(normalizar(ref, sin_tildes), normalizar(hyp, sin_tildes)))
    e = pd.DataFrame(filas, index=tabla.index, columns=["S", "D", "I", "N"])   # columnas fijas: tabla puede venir vacía
    e["WER"] = (e.S + e.D + e.I) / e.N.replace(0, np.nan)
    return e

def resumen_wer(tabla, metodos, por=None, sin_tildes=False):
    """WER agregado (suma de errores / suma de palabras) por grupo, con conteos."""
    tabla = tabla[tabla.Transcription.map(normalizar).str.len() > 0]
    claves = [por] if por else []
    base = tabla.groupby(claves) if por else tabla.assign(_g="TOTAL").groupby("_g")
    out = base.agg(segmentos=("Sample_ID", "size"), hablantes=("Speaker_ID", "nunique"))
    out["palabras"] = tabla.Transcription.map(lambda x: len(normalizar(x).split())).groupby(
        tabla[por] if por else pd.Series("TOTAL", index=tabla.index)).sum()
    for m in metodos:
        if m not in tabla or tabla[m].isna().all():
            continue
        e = wer_segmentos(tabla, m, sin_tildes)
        g = e.groupby(tabla[por] if por else pd.Series("TOTAL", index=tabla.index))
        s = g.sum()
        out[m] = (100 * (s.S + s.D + s.I) / s.N).round(1)
    return out

def _reducir_palabra(w):
    if len(w) <= 15:
        return w
    for L in range(1, len(w) // 2 + 1):
        if (w[:L] * (len(w) // L + 1))[:len(w)] == w:
            return w[:L]
    return w

def _clave(w):
    return re.sub(r"[^\w]", "", w.lower())

def postprocesar_reglas(texto, max_n=40):
    """Las 3 reglas del paper (sección 3.4). Solo borran: nunca agregan palabras."""
    toks = [_reducir_palabra(w) for w in str(texto).split()]
    out = []
    for w in toks:                                         # (ii) palabra repetida
        if out and _clave(w) and _clave(w) == _clave(out[-1]):
            continue
        out.append(w)
    toks, claves = out, [_clave(w) for w in out]
    cambio = True
    while cambio:                                          # (iii) frase repetida
        cambio = False
        for n in range(min(len(toks) // 2, max_n), 1, -1):
            i = 0
            while i + 2 * n <= len(toks):
                if claves[i:i + n] == claves[i + n:i + 2 * n]:
                    del toks[i + n:i + 2 * n]; del claves[i + n:i + 2 * n]; cambio = True
                else:
                    i += 1
    return " ".join(toks)

# Pruebas rápidas (si alguna falla, el notebook se detiene acá)
assert normalizar("¿Esto tiene tratamiento?\" y dice: \"no, no hay\".") == "esto tiene tratamiento y dice no no hay"
assert normalizar("Eso fue en el 20 y en 2009 <unk>") == "eso fue en el veinte y en dos mil nueve"
assert normalizar("Está aquí, señor", quitar_tildes=True) == "esta aqui señor"
assert postprocesar_reglas("y entonces y entonces y entonces me fui me fui") == "y entonces me fui"
assert postprocesar_reglas("jajajajajajajajajaja vale") == "ja vale"
assert contar_errores("a b c", "a x c d") == dict(S=1, D=0, I=1, N=3)
print("normalización, WER y reglas: OK")

## 4. Baseline A sobre todo el corpus

**Cómo transcribo:**
- Español forzado (`language="es"`, `task="transcribe"`) y decodificación **greedy**, como el paper.
- Segmentos ≤ 30 s: en lotes de 16. Para estos guardo además las **5 mejores hipótesis del beam search**, que usa el
  LLM en B.
- Segmentos > 30 s: el modo *long-form* de Whisper (avanza ventana a ventana usando timestamps), con
  `condition_on_prev_tokens=False` para que un error en una ventana no contamine la siguiente.

Los resultados se guardan en `resultados/pred_*.csv`. Si ya existen, se cargan en lugar de recalcular.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm.auto import tqdm

AUDIOS = {}
def cargar_audio(ruta):
    import soundfile as sf
    try:
        a, sr = sf.read(ruta, dtype="float32", always_2d=True)
        a = a.mean(axis=1)
        if sr == 16000:
            return a
    except Exception:
        pass
    import librosa
    a, _ = librosa.load(ruta, sr=16000, mono=True)
    return a.astype(np.float32)

for sid, ruta in zip(df.Sample_ID, df.ruta):
    AUDIOS[sid] = cargar_audio(ruta)

def cargar_whisper(nombre, adaptador=None):
    proc = WhisperProcessor.from_pretrained(nombre, language="spanish", task="transcribe")
    m = WhisperForConditionalGeneration.from_pretrained(nombre, torch_dtype=torch.float32)
    if adaptador:
        from peft import PeftModel
        m = PeftModel.from_pretrained(m, adaptador).merge_and_unload()
    m = m.to(DEVICE, DTYPE).eval()
    m.generation_config.forced_decoder_ids = None
    return proc, m

_TS = re.compile(r"<\|\d+\.\d+\|>")

@torch.inference_mode()
def transcribir_audios(audios, proc, model, nbest=0, lote=16, desc="transcribiendo"):
    gen = dict(language="es", task="transcribe")
    res = [None] * len(audios)
    cortos = [i for i, a in enumerate(audios) if len(a) <= 30 * 16000]
    largos = [i for i, a in enumerate(audios) if len(a) > 30 * 16000]
    barra = tqdm(total=len(audios), desc=desc)
    for k in range(0, len(cortos), lote):
        idx = cortos[k:k + lote]
        x = proc.feature_extractor([audios[i] for i in idx], sampling_rate=16000,
                                   return_tensors="pt").input_features.to(DEVICE, DTYPE)
        textos = proc.batch_decode(model.generate(x, max_new_tokens=400, num_beams=1, **gen),
                                   skip_special_tokens=True)
        alt = [[] for _ in idx]
        if nbest > 1:   # beam search en sub-lotes de 4: 16 audios x 5 haces no entra en memoria
            tn = []
            for s in range(0, len(idx), 4):
                tn += proc.batch_decode(model.generate(x[s:s + 4], max_new_tokens=400, num_beams=nbest,
                                                       num_return_sequences=nbest, **gen), skip_special_tokens=True)
            alt = [tn[j * nbest:(j + 1) * nbest] for j in range(len(idx))]
        for j, i in enumerate(idx):
            res[i] = dict(hyp=textos[j].strip(), nbest=[t.strip() for t in alt[j]])
        barra.update(len(idx))
    for i in largos:
        f = proc.feature_extractor(audios[i], sampling_rate=16000, return_tensors="pt", truncation=False,
                                   padding="longest", return_attention_mask=True)
        out = model.generate(f.input_features.to(DEVICE, DTYPE), attention_mask=f.attention_mask.to(DEVICE),
                             return_timestamps=True, condition_on_prev_tokens=False, **gen)
        texto = _TS.sub("", proc.batch_decode(out, skip_special_tokens=True)[0])
        res[i] = dict(hyp=" ".join(texto.split()), nbest=[])
        barra.update(1)
    barra.close()
    return res

def predecir(nombre_salida, sample_ids, proc, model, nbest=0):
    """Transcribe y guarda en resultados/pred_<nombre>.csv; si ya existe, lo carga."""
    ruta = os.path.join(RES, f"pred_{nombre_salida}.csv")
    if os.path.exists(ruta):
        p = pd.read_csv(ruta, keep_default_na=False)
        if set(sample_ids) <= set(p.Sample_ID):
            print("cargado de", ruta)
            p["nbest"] = p.nbest.map(json.loads)
            return p.set_index("Sample_ID").loc[list(sample_ids)].reset_index()
    t0 = time.time()
    r = transcribir_audios([AUDIOS[s] for s in sample_ids], proc, model, nbest=nbest, desc=nombre_salida)
    p = pd.DataFrame({"Sample_ID": list(sample_ids), "hyp": [x["hyp"] for x in r],
                      "nbest": [x["nbest"] for x in r]})
    p.assign(nbest=p.nbest.map(lambda v: json.dumps(v, ensure_ascii=False))).to_csv(ruta, index=False)
    print(f"{nombre_salida}: {len(p)} segmentos en {(time.time()-t0)/60:.1f} min → {ruta}")
    return p

proc_base, modelo_base = cargar_whisper(CFG["modelo_base"])
predA = predecir("A", df.Sample_ID.tolist(), proc_base, modelo_base, nbest=CFG["nbest"])
df["A"] = predA.hyp.values
df["A_nbest"] = predA.nbest.values
df["A+PP"] = df["A"].map(postprocesar_reglas)

### WER por segmento, de peor a mejor
`S/D/I` = sustituciones, borrados e inserciones. Un WER > 100 % significa que Whisper **inventó** más palabras de
las que había (alucinación).

In [ ]:
errA = wer_segmentos(df, "A")
segA = pd.concat([df[["Sample_ID", "Speaker_ID", "Disease", "Intelligibility", "Duration"]], errA,
                  wer_segmentos(df, "A+PP")[["WER"]].rename(columns={"WER": "WER_A+PP"})], axis=1)
segA = segA.sort_values("WER", ascending=False)
segA.to_csv(os.path.join(RES, "wer_por_segmento_A.csv"), index=False)
pd.set_option("display.max_colwidth", 140)
display(segA.head(25).round(2))
for _, r in segA.head(3).iterrows():
    fila = df.set_index("Sample_ID").loc[r.Sample_ID]
    print(f"\n[{r.Sample_ID}] WER {100*r.WER:.0f}%\n  REF: {fila.Transcription[:300]}\n  A  : {fila.A[:300]}")

for por in (None, "Disease", "Intelligibility"):
    display(resumen_wer(df, ["A", "A+PP"], por))

### Punto de decisión

Antes de invertir tiempo de GPU, esta celda resume qué tipo de errores comete A y qué sugiere eso para el resto:
- **Muchas inserciones** (alucinaciones) → las reglas y el LLM tienen margen, como mostró el paper.
- **Muchas sustituciones/borrados** → el problema es acústico: lo que puede ayudar es el fine-tuning (C).
- **WER total ya bajo** (< 10 %) → hay poco que ganar y la demo debería centrarse en los hablantes difíciles.

**Si querés cambiar el plan**, editá las banderas `CORRER_B`, `CORRER_C` y `CORRER_REF` de abajo y seguí.

In [ ]:
tot = errA.sum()
errores = tot.S + tot.D + tot.I
print(f"WER A (todo el corpus): {100*errores/tot.N:.1f}%  |  con reglas: {resumen_wer(df, ['A+PP'])['A+PP'].iloc[0]}%")
print(f"Reparto de errores → sustituciones {100*tot.S/errores:.0f}% | borrados {100*tot.D/errores:.0f}% | "
      f"inserciones {100*tot.I/errores:.0f}%")
print(f"Segmentos con WER > 100% (alucinación): {(errA.WER > 1).sum()} de {errA.WER.notna().sum()}")
if tot.I / errores > 0.35:
    print("→ Las inserciones pesan mucho: las reglas (PP) y el LLM tienen margen claro.")
if (tot.S + tot.D) / errores > 0.5:
    print("→ Más de la mitad de los errores son acústicos (S+D): el fine-tuning (C) es la apuesta principal.")
if 100 * errores / tot.N < 10:
    print("→ WER ya bajo: margen chico. Enfocá la demo en los hablantes de menor inteligibilidad.")

CORRER_B = CFG["llm_proveedor"] != "ninguno"
CORRER_C = DEVICE == "cuda"
CORRER_REF = CFG["evaluar_referencia"] and DEVICE == "cuda"
print(f"\nPlan: B={CORRER_B}  C={CORRER_C}  referencia grande={CORRER_REF}")

## 5. Splits por hablante

**Test (6 hablantes, 98 segmentos, ~24 min).** Criterios, en orden:
1. Ningún hablante compartido con entrenamiento ni validación.
2. Al menos un hablante por enfermedad, y los **5 niveles de inteligibilidad con ≥ 7 segmentos** cada uno.
3. Dejar en entrenamiento a los hablantes más grandes (ALS_M03, ALS_M05, STROKE_M01): en test dominarían el
   promedio y le quitarían mucho material al entrenamiento.
4. Dos hablantes de ACV en test porque es la enfermedad donde Whisper peor anda según el paper.

**Validación (4 hablantes, 22 segmentos)**, tomada del lado de entrenamiento pero **por hablante**: si fuera por
segmento, mediría cómo transcribe voces que ya escuchó y elegiría checkpoints con un criterio demasiado optimista.
Es chica, así que su señal es ruidosa: se usa solo para frenar el entrenamiento y elegir el mejor enfoque.

**Validación cruzada (opcional, `CFG["ejecutar_cv"]`).** Dejar afuera un hablante por vez serían 22 entrenamientos.
Uso 4 particiones por hablante con **exactamente un hablante de ACV cada una** (hay 4). Ventaja: el WER de C queda
medido sobre los 22 hablantes y no sobre 6. Costo: ~4 entrenamientos más (estimado 1–1,5 h en T4).

In [ ]:
TEST = ["ALS_M06", "ALS_F02", "ALS_M02", "PARKINSON_M01", "STROKE_F01", "STROKE_M02"]
VAL = ["ALS_F03", "ALS_M07", "PARKINSON_M03", "STROKE_M03"]
FOLDS = {
    0: ["STROKE_M01", "ALS_M02", "ALS_F03", "PARKINSON_M02", "PARKINSON_F03"],
    1: ["STROKE_F01", "ALS_M03", "ALS_M07", "PARKINSON_M01", "PARKINSON_F02"],
    2: ["STROKE_M02", "ALS_M04", "ALS_F01", "ALS_F02", "PARKINSON_F01", "PARKINSON_M04"],
    3: ["STROKE_M03", "ALS_M05", "ALS_M06", "ALS_M01", "PARKINSON_M03", "PARKINSON_F04"],
}
hablantes = set(df.Speaker_ID)
assert not set(TEST) & set(VAL)
assert set(TEST) | set(VAL) <= hablantes, "algún hablante del split no está en los datos"
assert sorted(sum(FOLDS.values(), [])) == sorted(hablantes) or len(faltan), "los folds no cubren a todos"

df["split"] = np.where(df.Speaker_ID.isin(TEST), "test", np.where(df.Speaker_ID.isin(VAL), "val", "train"))
df["fold_cv"] = df.Speaker_ID.map({s: k for k, v in FOLDS.items() for s in v})
df[["Sample_ID", "Speaker_ID", "Disease", "Intelligibility", "split", "fold_cv"]].to_csv(
    os.path.join(BASE, "splits.csv"), index=False)

display(df.groupby("split").agg(hablantes=("Speaker_ID", "nunique"), segmentos=("Sample_ID", "size"),
                                minutos=("Duration", lambda x: round(x.sum() / 60, 1))))
display(pd.crosstab(df.split, df.Intelligibility))
display(pd.crosstab(df.split, df.Disease))
for sp in ("train", "val", "test"):
    print(sp, sorted(df[df.split == sp].Speaker_ID.unique()))
assert not set(df[df.split == "train"].Speaker_ID) & set(df[df.split == "test"].Speaker_ID)
EVAL = df[df.split.isin(["val", "test"])].copy()

## 6. Enfoque B: corrección con LLM

**Qué recibe el LLM:** las hipótesis de Whisper **ya pasadas por las reglas**: la greedy primero y luego hasta 4
alternativas distintas del beam search (solo para segmentos ≤ 30 s; los largos llevan una sola hipótesis).
No ve la referencia ni el audio.

**Por qué así:** el paper mostró que borrar repeticiones es seguro. Un LLM es más potente pero también puede
"embellecer": devolver una frase fluida que la persona no dijo, lo que **sube** el WER. Para contenerlo:
1. El prompt pide ediciones mínimas, no parafrasear y conservar muletillas.
2. Le doy 3 ejemplos reales (hipótesis → referencia) de hablantes de **entrenamiento**: nunca de val/test.
3. **Salvaguarda:** si la respuesta cambia demasiado el largo (< 50 % o > 150 % + 3 palabras) me quedo con la
   hipótesis original y lo cuento.
4. Las respuestas se guardan en caché (`resultados/cache_llm.json`): reanudar no vuelve a pagar llamadas.

Con Anthropic uso el SDK oficial, `claude-opus-5-5` con esfuerzo bajo (es una tarea corta) y
`fallbacks="default"`: si el modelo declina un pedido, la API lo reintenta en otro modelo. El costo estimado para
~120 segmentos (val+test) × 2 enfoques es de pocos dólares; para abaratar podés poner `llm_modelo="claude-haiku-5-5"`.

In [ ]:
SYSTEM_LLM = """Corregís transcripciones automáticas (ASR) de habla espontánea en español producida por personas con disartria (ELA, Parkinson o secuelas de ACV). Recibís una o varias hipótesis del reconocedor para el MISMO audio, ordenadas de más a menos probable.

Devolvé la transcripción más probable de lo que la persona dijo.
- Corregí solo errores típicos de reconocimiento: palabras que suenan parecido, palabras inexistentes, cortes o uniones de palabras, repeticiones en bucle del reconocedor.
- Usá las hipótesis alternativas para decidir entre palabras dudosas.
- No agregues información que no esté en ninguna hipótesis, no resumas, no parafrasees y no corrijas la gramática ni el estilo de la persona.
- Mantené muletillas y repeticiones naturales ("eh", "o sea", "no, no").
- Si la primera hipótesis parece correcta, devolvela sin cambios. Ante la duda, cambiá lo menos posible.
- Respondé solo con la transcripción final, sin comillas ni comentarios."""

def obtener_secreto(nombre):
    if os.environ.get(nombre):
        return os.environ[nombre]
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nombre)
    except Exception:
        return None

_cliente = {}
def _llm_anthropic(mensajes):
    import anthropic
    if "a" not in _cliente:
        _cliente["a"] = anthropic.Anthropic(api_key=obtener_secreto("ANTHROPIC_API_KEY"), max_retries=4)
    kw = dict(model=CFG["llm_modelo"], max_tokens=4000, system=SYSTEM_LLM, messages=mensajes)
    if CFG["llm_esfuerzo"]:
        kw["output_config"] = {"effort": CFG["llm_esfuerzo"]}
    if CFG["llm_modelo"] in ("claude-opus-5-5", "claude-opus-5", "claude-fable-5-1", "claude-sonnet-5-5"):
        r = _cliente["a"].beta.messages.create(betas=["server-side-fallback-2026-07-01"], fallbacks="default", **kw)
    else:
        r = _cliente["a"].messages.create(**kw)
    if r.stop_reason == "refusal":
        return None
    return "".join(b.text for b in r.content if b.type == "text")

def nombre_clave():
    if CFG["llm_proveedor"] == "anthropic":
        return "ANTHROPIC_API_KEY"
    return "GROQ_API_KEY" if "groq" in (CFG["llm_base_url"] or "") else "OPENAI_API_KEY"

def _llm_openai(mensajes):
    from openai import OpenAI
    if "o" not in _cliente:
        _cliente["o"] = OpenAI(api_key=obtener_secreto(nombre_clave()) or "sin-clave",
                               base_url=CFG["llm_base_url"], max_retries=0)
    r = _cliente["o"].chat.completions.create(
        model=CFG["llm_modelo"], messages=[{"role": "system", "content": SYSTEM_LLM}] + mensajes)
    texto = r.choices[0].message.content or ""
    return re.sub(r"<think>.*?</think>", "", texto, flags=re.S)   # modelos que razonan en voz alta

ERRORES_DEFINITIVOS = {"NotFoundError", "AuthenticationError", "PermissionDeniedError", "BadRequestError",
                       "UnprocessableEntityError"}

def probar_llm():
    """Una llamada de prueba antes de lanzar todas. Devuelve True si el proveedor y el modelo responden."""
    f = {"anthropic": _llm_anthropic, "openai_compatible": _llm_openai}[CFG["llm_proveedor"]]
    try:
        f([{"role": "user", "content": "Hipótesis 1: hola buenos días"}])
        print(f"LLM OK: {CFG['llm_proveedor']} / {CFG['llm_modelo']}")
        return True
    except Exception as e:
        print(f"El LLM no responde ({type(e).__name__}): {str(e)[:300]}")
        if type(e).__name__ == "NotFoundError":
            print(f"El modelo '{CFG['llm_modelo']}' no existe en el proveedor. Modelos disponibles:")
            try:
                from openai import OpenAI
                if CFG["llm_proveedor"] == "openai_compatible":
                    cli = OpenAI(api_key=obtener_secreto(nombre_clave()), base_url=CFG["llm_base_url"])
                    print(sorted(m.id for m in cli.models.list().data))
            except Exception:
                pass
            print('Elegí uno grande de propósito general y poné CFG["llm_modelo"] = "..." en la configuración.')
        elif type(e).__name__ in ("AuthenticationError", "PermissionDeniedError"):
            print(f"Revisá el secreto {nombre_clave()}: clave inválida o sin 'Acceso al notebook'.")
        return False

RUTA_CACHE = os.path.join(RES, "cache_llm.json")
CACHE_LLM = json.load(open(RUTA_CACHE, encoding="utf-8")) if os.path.exists(RUTA_CACHE) else {}

def llamar_llm(mensajes):
    clave = hashlib.sha1(json.dumps([CFG["llm_proveedor"], CFG["llm_modelo"], SYSTEM_LLM, mensajes],
                                    ensure_ascii=False).encode()).hexdigest()
    if clave in CACHE_LLM:
        return CACHE_LLM[clave]
    f = {"anthropic": _llm_anthropic, "openai_compatible": _llm_openai}[CFG["llm_proveedor"]]
    for intento in range(6):   # reintentos con espera creciente: los planes gratuitos limitan llamadas por minuto
        try:
            salida = f(mensajes)
            break
        except Exception as e:
            # Modelo inexistente, clave inválida o pedido mal formado: reintentar no sirve
            if intento == 5 or type(e).__name__ in ERRORES_DEFINITIVOS:
                raise
            espera = min(60, 5 * 2 ** intento)
            print(f"LLM: {type(e).__name__}, reintento en {espera}s")
            time.sleep(espera)
    CACHE_LLM[clave] = salida
    return salida

def formatear_hipotesis(hyps):
    return "\n".join(f"Hipótesis {i+1}: {h}" for i, h in enumerate(hyps))

def hipotesis_para_llm(greedy, nbest):
    vistas, hyps = set(), []
    for h in [greedy] + list(nbest):
        h = postprocesar_reglas(h)
        if h.strip() and normalizar(h) not in vistas:
            vistas.add(normalizar(h)); hyps.append(h)
    return hyps[:CFG["nbest"]] or [greedy]

def elegir_ejemplos(k):
    """Ejemplos few-shot de hablantes de ENTRENAMIENTO, uno por enfermedad, con errores moderados."""
    tr = df[(df.split == "train") & (df.Duration.between(5, 20))].copy()
    tr["wer"] = wer_segmentos(tr, "A").WER
    tr = tr[tr.wer.between(0.15, 0.6)].sort_values("wer")
    ej = tr.groupby("Disease").head(1).head(k)
    msgs = []
    for _, r in ej.iterrows():
        msgs += [{"role": "user", "content": formatear_hipotesis(hipotesis_para_llm(r.A, r.A_nbest))},
                 {"role": "assistant", "content": r.Transcription.replace("<unk>", "").strip()}]
    return msgs, ej.Sample_ID.tolist()

N_DESCARTADAS = {"n": 0}
def corregir(greedy, nbest, ejemplos):
    hyps = hipotesis_para_llm(greedy, nbest)
    base = hyps[0]
    if not base.strip():
        return base
    try:
        salida = llamar_llm(ejemplos + [{"role": "user", "content": formatear_hipotesis(hyps)}])
    except Exception as e:
        N_DESCARTADAS["fallos"] = N_DESCARTADAS.get("fallos", 0) + 1
        print("Error de LLM:", type(e).__name__, str(e)[:200])
        return base
    salida = (salida or "").strip().strip('"“”«»').strip()
    nb, ns = len(base.split()), len(salida.split())
    if not salida or ns < 0.5 * nb or ns > 1.5 * nb + 3:
        N_DESCARTADAS["n"] += 1
        return base
    return salida

def aplicar_llm(tabla, col_greedy, col_nbest, ejemplos):
    from concurrent.futures import ThreadPoolExecutor
    filas = list(zip(tabla[col_greedy], tabla[col_nbest]))
    N_DESCARTADAS["fallos"] = 0
    with ThreadPoolExecutor(CFG["llm_hilos"]) as ex:
        out = list(tqdm(ex.map(lambda f: corregir(f[0], f[1], ejemplos), filas), total=len(filas), desc="LLM"))
    json.dump(CACHE_LLM, open(RUTA_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    if N_DESCARTADAS["fallos"]:
        print(f"ATENCIÓN: {N_DESCARTADAS['fallos']} de {len(filas)} llamadas fallaron y quedaron SIN corregir. "
              "Volvé a correr la celda: las exitosas salen de la caché y solo se reintentan las fallidas.")
    return out

if CORRER_B and not obtener_secreto(nombre_clave()):
    print(f"No encuentro {nombre_clave()}: salto B y D. Cargala como secreto y volvé a correr esta celda.")
    CORRER_B = False
if CORRER_B and not probar_llm():
    print("Salto B y D. Corregí la configuración y volvé a correr desde esta celda.")
    CORRER_B = False

EVAL = df[df.split.isin(["val", "test"])].copy()
if CORRER_B:
    EJEMPLOS, ids_ej = elegir_ejemplos(CFG["llm_ejemplos"])
    assert not set(ids_ej) & set(EVAL.Sample_ID)
    print("Ejemplos few-shot (train):", ids_ej)
    EVAL["B"] = aplicar_llm(EVAL, "A", "A_nbest", EJEMPLOS)
    print(f"Respuestas descartadas por la salvaguarda de largo: {N_DESCARTADAS['n']}")
    display(resumen_wer(EVAL[EVAL.split == "test"], ["A", "A+PP", "B"]))

### Ejemplos antes/después (test)
Segmentos donde el LLM cambió algo, con el WER de cada versión. Revisalos: es la mejor forma de detectar si está
"inventando".

In [ ]:
if CORRER_B:
    t = EVAL[(EVAL.split == "test") & (EVAL["B"].map(normalizar) != EVAL["A+PP"].map(normalizar))].copy()
    t["WER A+PP"] = wer_segmentos(t, "A+PP").WER
    t["WER B"] = wer_segmentos(t, "B").WER
    print(f"El LLM modificó {len(t)} de {(EVAL.split == 'test').sum()} segmentos de test. "
          f"Mejoró {(t['WER B'] < t['WER A+PP']).sum()}, empeoró {(t['WER B'] > t['WER A+PP']).sum()}.")
    for _, r in t.sample(min(8, len(t)), random_state=0).iterrows():
        print(f"\n[{r.Sample_ID} | {r.Disease}] WER {100*r['WER A+PP']:.0f}% → {100*r['WER B']:.0f}%")
        print("  REF :", r.Transcription[:250]); print("  A+PP:", r["A+PP"][:250]); print("  B   :", r.B[:250])

## 7. Fine-tuning con LoRA (C)

### 7.1 Trocear los segmentos largos
Whisper aprende con audios de hasta 30 s, pero el 34 % de los segmentos de entrenamiento dura más. Descartarlos
tiraría la mayor parte de las horas. Para cortarlos **sin perder la correspondencia audio↔texto** hago
**alineación forzada**:
1. Un modelo CTC de español (`wav2vec2-large-xlsr-53-spanish`) da, cada 20 ms, la probabilidad de cada letra.
2. Como ya conozco el texto correcto, busco con programación dinámica (Viterbi) en qué momento se dice cada letra.
   Esto es mucho más robusto que *reconocer*: el modelo solo tiene que ubicar un texto conocido.
3. Corto entre palabras, en el silencio entre el fin de una y el inicio de la siguiente, en trozos de ≤ 28 s.

Si la alineación de un segmento falla (por ejemplo, el audio es más corto que el texto), ese segmento se descarta
del entrenamiento y lo informo. Abajo podés escuchar un trozo para verificar que el texto coincide.

In [ ]:
def cargar_alineador():
    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
    p = Wav2Vec2Processor.from_pretrained(CFG["alineador"])
    m = Wav2Vec2ForCTC.from_pretrained(CFG["alineador"]).to(DEVICE, DTYPE).eval()
    return p, m

@torch.inference_mode()
def emisiones(audio, p, m, ventana_s=30):
    """log-probabilidades CTC (frames x vocab) y el tiempo en segundos de cada frame."""
    lps, tiempos = [], []
    paso = ventana_s * 16000
    for ini in range(0, len(audio), paso):
        trozo = audio[ini:ini + paso]
        if len(trozo) < 800:
            trozo = np.pad(trozo, (0, 800 - len(trozo)))
        x = p(trozo, sampling_rate=16000, return_tensors="pt").input_values.to(DEVICE, DTYPE)
        lp = m(x).logits[0].float().log_softmax(-1).cpu()
        lps.append(lp)
        tiempos.append(ini / 16000 + 0.02 * np.arange(lp.shape[0]))
    return torch.cat(lps), np.concatenate(tiempos)

def palabras_para_alinear(transcripcion, vocab):
    """[(palabra_original, [ids de letras])]. La puntuación suelta se pega a la palabra vecina."""
    items, pendiente = [], ""
    for w in transcripcion.replace("<unk>", " ").split():
        letras = [vocab[c] for c in normalizar(w).replace(" ", "") if c in vocab]
        if letras:
            items.append([(pendiente + " " + w).strip(), letras]); pendiente = ""
        elif items:
            items[-1][0] += " " + w
        else:
            pendiente += " " + w
    return items

def alinear_ctc(lp, tokens, blank):
    """Viterbi CTC simplificado. Devuelve el frame donde empieza cada token, o None."""
    T, L = lp.shape[0], len(tokens)
    if T < L:
        return None
    tok = torch.tensor(tokens)
    tr = torch.full((T + 1, L + 1), -float("inf"))
    tr[0, 0] = 0.0
    for t in range(T):
        quedarse = torch.cat([lp[t, blank:blank + 1], torch.maximum(lp[t, blank].expand(L), lp[t, tok])])
        avanzar = lp[t, tok]
        tr[t + 1, 0] = tr[t, 0] + quedarse[0]
        tr[t + 1, 1:] = torch.maximum(tr[t, 1:] + quedarse[1:], tr[t, :-1] + avanzar)
    if not torch.isfinite(tr[T, L]):
        return None
    inicio, j = [0] * L, L
    for t in range(T, 0, -1):
        if j == 0:
            break
        q = tr[t - 1, j] + max(lp[t - 1, blank].item(), lp[t - 1, tokens[j - 1]].item())
        a = tr[t - 1, j - 1] + lp[t - 1, tokens[j - 1]].item()
        if a >= q:
            inicio[j - 1] = t - 1; j -= 1
    return inicio if j == 0 else None

def trocear_segmento(sid, texto, p, m, max_s):
    audio = AUDIOS[sid]
    vocab = p.tokenizer.get_vocab()
    sep, blank = vocab.get("|"), p.tokenizer.pad_token_id
    items = palabras_para_alinear(texto, vocab)
    if not items:
        return []
    tokens, rangos = [], []
    for k, (_, letras) in enumerate(items):
        if k and sep is not None:
            tokens.append(sep)
        rangos.append((len(tokens), len(tokens) + len(letras) - 1))
        tokens += letras
    lp, tiempos = emisiones(audio, p, m)
    ini = alinear_ctc(lp, tokens, blank)
    if ini is None:
        return None
    pal = [(w, tiempos[ini[a]], tiempos[ini[b]] + 0.02) for (w, _), (a, b) in zip(items, rangos)]
    trozos, actual, t0 = [], [], 0.0
    for k, (w, wi, wf) in enumerate(pal):
        if actual and wf - t0 > max_s:
            corte = (pal[k - 1][2] + wi) / 2
            trozos.append((t0, corte, " ".join(x[0] for x in actual)))
            actual, t0 = [], corte
        actual.append((w, wi, wf))
    trozos.append((t0, len(audio) / 16000, " ".join(x[0] for x in actual)))
    return [(sid, a, b, t) for a, b, t in trozos if 1.0 <= b - a <= 30.0 and t.strip()]

def tabla_de_trozos(sample_ids):
    """Segmentos <=30 s enteros; los largos, troceados. Se guarda en Drive para no repetir."""
    ruta = os.path.join(BASE, "trozos_entrenamiento.csv")
    previos = pd.read_csv(ruta, keep_default_na=False) if os.path.exists(ruta) else pd.DataFrame(
        columns=["Sample_ID", "ini", "fin", "texto", "origen"])
    hechos = set(previos.Sample_ID)
    nuevos, fallidos, p, m = [], [], None, None
    sub = df[df.Sample_ID.isin(sample_ids) & ~df.Sample_ID.isin(hechos)]
    for _, r in tqdm(list(sub.iterrows()), desc="alineando"):
        if r.Duration <= 30:
            nuevos.append((r.Sample_ID, 0.0, len(AUDIOS[r.Sample_ID]) / 16000,
                           " ".join(r.Transcription.replace("<unk>", " ").split()), "entero"))
            continue
        if p is None:
            p, m = cargar_alineador()
        tz = trocear_segmento(r.Sample_ID, r.Transcription, p, m, CFG["max_trozo_s"])
        if not tz:
            fallidos.append(r.Sample_ID); continue
        nuevos += [t + ("troceado",) for t in tz]
    if m is not None:
        del m; torch.cuda.empty_cache()
    tabla = pd.concat([previos, pd.DataFrame(nuevos, columns=previos.columns)], ignore_index=True)
    tabla = tabla[tabla.texto.str.strip().str.len() > 0]
    tabla.to_csv(ruta, index=False)
    if fallidos:
        print("No se pudieron alinear (se excluyen del entrenamiento):", fallidos)
    return tabla[tabla.Sample_ID.isin(sample_ids)].reset_index(drop=True)

if CORRER_C:
    TROZOS = tabla_de_trozos(df[df.split.isin(["train", "val"]) | CFG["ejecutar_cv"]].Sample_ID.tolist())
    TROZOS = TROZOS.merge(df[["Sample_ID", "Speaker_ID", "split"]], on="Sample_ID")
    display(TROZOS.assign(minutos=(TROZOS.fin - TROZOS.ini) / 60).groupby(["split", "origen"]).agg(
        trozos=("Sample_ID", "size"), minutos=("minutos", "sum")).round(1))
    from IPython.display import Audio, display as _d
    ej = TROZOS[TROZOS.origen == "troceado"].sample(1, random_state=1).iloc[0]
    print(f"Escuchá este trozo ({ej.fin-ej.ini:.1f} s) y compará con el texto:\n  {ej.texto}")
    _d(Audio(AUDIOS[ej.Sample_ID][int(ej.ini * 16000):int(ej.fin * 16000)], rate=16000))

### 7.2 Configuración del entrenamiento y por qué

| Parámetro | Valor | Por qué |
|---|---|---|
| modelo | `whisper-small` (244 M) | Entra holgado en una T4 y entrena en minutos. `large` en T4 con LoRA es posible pero ~5× más lento. |
| idioma / tarea | `es` / `transcribe` | Prefijo fijo en las etiquetas y en la generación, así el modelo no "elige" idioma. |
| LoRA `r` | 16 | Rango de las matrices de adaptación (~3,5 M parámetros entrenables, 1,4 %). El paper usó 8 en un modelo 6× más grande; 16 da algo más de capacidad sin acercarse a ajustar todo. |
| `lora_alpha` | 32 | Escala de la actualización (`alpha/r` = 2, convención habitual). |
| `lora_dropout` | 0,1 | Regularización: hay solo ~2,7 h y 12 hablantes. |
| módulos | `q,k,v,out_proj,fc1,fc2` en encoder y decoder | El encoder tiene que adaptarse a la **acústica** de la disartria; adaptar solo atención no alcanza. |
| `lr` | 5e-4 | Valor típico para LoRA (el paper usó 3e-4 con un modelo más grande). Con fine-tuning completo sería ~1e-5. |
| `batch` × `grad_accum` | 8 × 2 = 16 | Lote efectivo 16, igual que el paper. 8 entra en la T4 en fp16 sin *gradient checkpointing*. |
| `fp16` | sí | La T4 tiene núcleos para fp16; los pesos LoRA se mantienen en fp32 (precisión mixta). |
| épocas | hasta 8 | Con *early stopping*: se frena si la pérdida en validación no mejora en 3 épocas. |
| `warmup` | 10 % de los pasos | Evita pasos grandes al inicio. El paper usó 100 pasos fijos, pero acá eso serían ~2 épocas. |
| scheduler | coseno | Baja el lr suavemente, como el paper. |
| SpecAugment | `mask_time_prob=0.05` | Tapa franjas de tiempo del espectrograma al entrenar: regularización barata para datos chicos. |
| selección | menor pérdida de validación | `load_best_model_at_end`: se queda con la mejor época, no con la última. |

**Checkpoints y reanudación.** Se guarda un checkpoint por época (solo el adaptador LoRA, ~25 MB) en
`BASE/lora/...`. Si la sesión se corta, al volver a correr esta sección `trainer.train(resume_from_checkpoint=True)`
continúa desde el último. Al terminar se escribe `LISTO.json`; si existe, el entrenamiento se salta.

**TORGO (no lo hago).** El paper entrenó con TORGO + NeuroVoz y **empeoró** Whisper en S-DiverSe: palabras aisladas
en inglés grabadas en laboratorio no se parecen a entrevistas espontáneas en español, y el modelo olvida lo que
sabía. Preentrenar con TORGO solo valdría la pena como experimento comparativo, si sobra tiempo.

In [ ]:
from dataclasses import dataclass
from peft import LoraConfig, get_peft_model
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

class DatasetTrozos(torch.utils.data.Dataset):
    def __init__(self, tabla, proc):
        self.t = tabla.reset_index(drop=True); self.proc = proc
    def __len__(self):
        return len(self.t)
    def __getitem__(self, i):
        r = self.t.iloc[i]
        a = AUDIOS[r.Sample_ID][int(r.ini * 16000):int(r.fin * 16000)]
        return {"input_features": self.proc.feature_extractor(a, sampling_rate=16000).input_features[0],
                "labels": self.proc.tokenizer(r.texto).input_ids}

@dataclass
class Collator:
    proc: object
    inicio: int
    def __call__(self, items):
        x = self.proc.feature_extractor.pad([{"input_features": f["input_features"]} for f in items],
                                            return_tensors="pt")
        lab = self.proc.tokenizer.pad([{"input_ids": f["labels"]} for f in items], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab["attention_mask"].ne(1), -100)
        if (labels[:, 0] == self.inicio).all():
            labels = labels[:, 1:]          # el modelo agrega <|startoftranscript|> solo
        x["labels"] = labels
        return x

def crear_modelo_lora():
    m = WhisperForConditionalGeneration.from_pretrained(CFG["modelo_base"], torch_dtype=torch.float32)
    m.config.use_cache = False
    m.config.apply_spec_augment, m.config.mask_time_prob = True, 0.05
    m.generation_config.language, m.generation_config.task = "es", "transcribe"
    m.generation_config.forced_decoder_ids = None
    lc = LoraConfig(r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"], bias="none",
                    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"])
    m = get_peft_model(m, lc)
    m.print_trainable_parameters()
    return m

def entrenar_lora(hablantes_train, hablantes_val, nombre, max_pasos=-1):
    salida = os.path.join(BASE, "lora", nombre)
    final = os.path.join(salida, "adaptador_final")
    if os.path.exists(os.path.join(salida, "LISTO.json")):
        print("Ya entrenado:", final); return final
    fijar_semilla()
    proc = WhisperProcessor.from_pretrained(CFG["modelo_base"], language="spanish", task="transcribe")
    tr = TROZOS[TROZOS.Speaker_ID.isin(hablantes_train)]
    va = TROZOS[TROZOS.Speaker_ID.isin(hablantes_val)]
    largo = tr.texto.map(lambda s: len(proc.tokenizer(s).input_ids))
    if (largo > 440).any():
        print(f"Descarto {(largo > 440).sum()} trozos con más de 440 tokens"); tr = tr[largo <= 440]
    print(f"[{nombre}] train: {len(tr)} trozos de {tr.Speaker_ID.nunique()} hablantes | val: {len(va)} trozos")
    args = Seq2SeqTrainingArguments(
        output_dir=salida, per_device_train_batch_size=CFG["batch"], per_device_eval_batch_size=CFG["batch"],
        gradient_accumulation_steps=CFG["grad_accum"], learning_rate=CFG["lr"], num_train_epochs=CFG["epocas"],
        max_steps=max_pasos, lr_scheduler_type="cosine", warmup_ratio=CFG["warmup_ratio"],
        fp16=(DEVICE == "cuda"), eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        logging_steps=5, report_to="none", remove_unused_columns=False, label_names=["labels"],
        dataloader_num_workers=2 if os.name != "nt" else 0, seed=CFG["seed"])
    trainer = Seq2SeqTrainer(
        model=crear_modelo_lora(), args=args, train_dataset=DatasetTrozos(tr, proc),
        eval_dataset=DatasetTrozos(va, proc),
        data_collator=Collator(proc, proc.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG["paciencia"])])
    hay_ckpt = bool(glob.glob(os.path.join(salida, "checkpoint-*")))
    if hay_ckpt:
        print("Reanudando desde el último checkpoint")
    t0 = time.time()
    trainer.train(resume_from_checkpoint=hay_ckpt)
    trainer.model.save_pretrained(final)
    hist = pd.DataFrame(trainer.state.log_history)
    hist.to_csv(os.path.join(salida, "historial.csv"), index=False)
    json.dump({"minutos": (time.time() - t0) / 60, "mejor_checkpoint": trainer.state.best_model_checkpoint,
               "mejor_eval_loss": trainer.state.best_metric}, open(os.path.join(salida, "LISTO.json"), "w"))
    print(f"Entrenado en {(time.time()-t0)/60:.1f} min. Mejor: {trainer.state.best_model_checkpoint}")
    if {"loss", "eval_loss"} <= set(hist.columns):
        ax = hist.dropna(subset=["loss"]).plot(x="epoch", y="loss", figsize=(6, 3))
        hist.dropna(subset=["eval_loss"]).plot(x="epoch", y="eval_loss", ax=ax, marker="o"); plt.show()
    del trainer; torch.cuda.empty_cache()
    return final

if CORRER_C:
    TRAIN_SPK = sorted(df[df.split == "train"].Speaker_ID.unique())
    ADAPTADOR = entrenar_lora(TRAIN_SPK, VAL, "principal")

### 7.3 Evaluar C
El adaptador se fusiona con el modelo base (`merge_and_unload`), así la inferencia es igual de rápida que A.
Transcribo val+test con las mismas opciones que A, incluidas las 5 hipótesis para D.

In [ ]:
if CORRER_C:
    proc_C, modelo_C = cargar_whisper(CFG["modelo_base"], ADAPTADOR)
    predC = predecir("C", EVAL.Sample_ID.tolist(), proc_C, modelo_C, nbest=CFG["nbest"])
    EVAL["C"] = predC.hyp.values
    EVAL["C_nbest"] = predC.nbest.values
    EVAL["C+PP"] = EVAL["C"].map(postprocesar_reglas)
    display(resumen_wer(EVAL[EVAL.split == "test"], ["A", "A+PP", "C", "C+PP"]))

### 7.4 (Opcional) Validación cruzada de C por hablante
Solo corre con `CFG["ejecutar_cv"] = True`. Cada fold entrena un adaptador sin esos hablantes y los transcribe.
La validación de cada fold son los hablantes de `VAL` que no caen en el fold (más uno de reemplazo si hace falta).
Al final, **los 22 hablantes** tienen una transcripción de C hecha por un modelo que nunca los escuchó.
Es reanudable fold por fold.

In [ ]:
if CORRER_C and CFG["ejecutar_cv"]:
    cv = []
    for k, fold_spk in FOLDS.items():
        resto = [s for s in sorted(hablantes) if s not in fold_spk]
        val_k = [s for s in VAL if s in resto]
        for enf in ("ALS", "PARKINSON", "STROKE"):
            if not any(s.startswith(enf) for s in val_k):
                cand = df[df.Speaker_ID.isin(resto) & (df.Disease == enf)].groupby("Speaker_ID").size()
                cand = cand[cand >= 2].sort_values()
                if len(cand):
                    val_k.append(cand.index[0])
        tr_k = [s for s in resto if s not in val_k]
        ad = entrenar_lora(tr_k, val_k, f"cv_fold{k}")
        p_k, m_k = cargar_whisper(CFG["modelo_base"], ad)
        ids = df[df.Speaker_ID.isin(fold_spk)].Sample_ID.tolist()
        cv.append(predecir(f"C_cv_fold{k}", ids, p_k, m_k))
        del m_k; torch.cuda.empty_cache()
    pcv = pd.concat(cv)
    df_cv = df.merge(pcv[["Sample_ID", "hyp"]].rename(columns={"hyp": "C_cv"}), on="Sample_ID")
    df_cv["C_cv+PP"] = df_cv["C_cv"].map(postprocesar_reglas)
    print("Validación cruzada: los 22 hablantes, cada uno evaluado por un modelo que no lo vio")
    for por in (None, "Disease", "Intelligibility", "Speaker_ID"):
        display(resumen_wer(df_cv, ["A", "A+PP", "C_cv", "C_cv+PP"], por))

## 8. Enfoque D y tabla final

**D** = C → reglas → LLM, con los mismos ejemplos y salvaguardas que B.

**Cómo elijo el "mejor enfoque"**: por WER en **validación**, no en test. Elegir mirando test sería hacer trampa
a pequeña escala (el número final quedaría optimista). Test se usa solo para reportar.

**Incertidumbre.** Con 6 hablantes de test, el número depende mucho de quiénes son. Calculo un intervalo de 95 %
por *bootstrap* **de hablantes** (remuestreo hablantes completos, no segmentos) para la diferencia A − mejor. Si el
intervalo no cruza 0, la mejora no se explica por un hablante afortunado.

In [ ]:
if CORRER_C and CORRER_B:
    EVAL["D"] = aplicar_llm(EVAL, "C", "C_nbest", EJEMPLOS)
if CORRER_REF:
    proc_R, modelo_R = cargar_whisper(CFG["modelo_referencia"])
    EVAL["Ref. large-v3-turbo"] = predecir("ref_turbo", EVAL.Sample_ID.tolist(), proc_R, modelo_R).hyp.values
    del modelo_R; torch.cuda.empty_cache()

METODOS = [m for m in ["A", "A+PP", "B", "C", "C+PP", "D", "Ref. large-v3-turbo"] if m in EVAL]
CANDIDATOS = [m for m in ["A+PP", "B", "C", "C+PP", "D"] if m in EVAL]
wer_val = resumen_wer(EVAL[EVAL.split == "val"], CANDIDATOS).iloc[0][CANDIDATOS]
MEJOR = wer_val.astype(float).idxmin()
print("WER en validación:", wer_val.to_dict(), "→ mejor enfoque:", MEJOR)

TEST_DF = EVAL[EVAL.split == "test"]
tablas = {}
for nombre, por in [("total", None), ("por_enfermedad", "Disease"), ("por_inteligibilidad", "Intelligibility"),
                    ("por_hablante", "Speaker_ID")]:
    tablas[nombre] = resumen_wer(TEST_DF, METODOS, por)
    tablas[nombre].to_csv(os.path.join(RES, f"tabla_{nombre}.csv"))
    print(f"\n=== WER (%) en TEST, {nombre.replace('_', ' ')} ===")
    display(tablas[nombre])
print("\nSecundario: WER sin tildes (TEST)")
display(resumen_wer(TEST_DF, METODOS, sin_tildes=True))

filas = []
for m in METODOS:
    s = wer_segmentos(TEST_DF, m).sum()
    filas.append(dict(metodo=m, sustituciones=int(s.S), borrados=int(s.D), inserciones=int(s.I), palabras=int(s.N)))
display(pd.DataFrame(filas).set_index("metodo"))

def bootstrap_hablantes(tabla, m1, m2, n=2000, seed=0):
    e1, e2 = wer_segmentos(tabla, m1), wer_segmentos(tabla, m2)
    g = pd.DataFrame({"spk": tabla.Speaker_ID, "err1": e1.S + e1.D + e1.I, "err2": e2.S + e2.D + e2.I,
                      "N": e1.N}).groupby("spk").sum()
    rng = np.random.default_rng(seed)
    difs = []
    for _ in range(n):
        s = g.iloc[rng.integers(0, len(g), len(g))]
        difs.append(100 * (s.err1.sum() - s.err2.sum()) / s.N.sum())
    return np.percentile(difs, [2.5, 50, 97.5])

lo, med, hi = bootstrap_hablantes(TEST_DF, "A", MEJOR)
print(f"\nMejora A − {MEJOR}: {med:.1f} puntos de WER (IC 95% por hablantes: {lo:.1f} a {hi:.1f}; "
      f"{TEST_DF.Speaker_ID.nunique()} hablantes, {len(TEST_DF)} segmentos)")
EVAL.drop(columns=[c for c in EVAL if c.endswith("nbest")]).to_csv(
    os.path.join(RES, "transcripciones_val_test.csv"), index=False)

## 9. Audios para la demo
5 segmentos de **test** (hablantes que ningún modelo vio) donde A falla más y el mejor enfoque acierta.
Criterio: mayor diferencia de WER, con el mejor enfoque ≤ 35 % de WER, duración ≤ 30 s (cómodo para mostrar) y, si se
puede, hablantes distintos. Se copian a `BASE/demo/` con un CSV de transcripciones.

In [ ]:
from IPython.display import Audio
t = TEST_DF.copy()
t["WER_A"] = wer_segmentos(t, "A").WER
t["WER_mejor"] = wer_segmentos(t, MEJOR).WER
t = t[(t.Duration <= 30) & (t.WER_mejor <= 0.35)].assign(ganancia=lambda x: x.WER_A - x.WER_mejor)
t = t.sort_values("ganancia", ascending=False)
demo = t.groupby("Speaker_ID").head(1).head(5)
if len(demo) < 5:
    demo = pd.concat([demo, t[~t.Sample_ID.isin(demo.Sample_ID)].head(5 - len(demo))])
DEMO_DIR = os.path.join(BASE, "demo"); os.makedirs(DEMO_DIR, exist_ok=True)
for _, r in demo.iterrows():
    shutil.copy(r.ruta, DEMO_DIR)
    print(f"\n[{r.Sample_ID} | {r.Disease} | {r.Intelligibility}] WER A {100*r.WER_A:.0f}% → {MEJOR} {100*r.WER_mejor:.0f}%")
    print("  Referencia:", r.Transcription)
    print("  A (Whisper):", r.A)
    print(f"  {MEJOR}:", r[MEJOR])
    display(Audio(r.ruta))
demo[["Sample_ID", "Speaker_ID", "Disease", "Intelligibility", "Transcription", "A", MEJOR, "WER_A", "WER_mejor"]].to_csv(
    os.path.join(DEMO_DIR, "demo.csv"), index=False)
print("\nCopiados a", DEMO_DIR)

## 10. Inferencia: `transcribir(ruta_audio)`
Acepta cualquier audio (wav, mp3, m4a, ogg...: se convierte a 16 kHz mono) y devuelve A y el mejor enfoque lado a lado.

In [ ]:
_MODELOS = {}
def _modelo(nombre):
    if nombre not in _MODELOS:
        if nombre == "A":
            _MODELOS[nombre] = (proc_base, modelo_base)
        else:
            _MODELOS[nombre] = (proc_C, modelo_C)
    return _MODELOS[nombre]

def transcribir(ruta_audio, mejor=None):
    mejor = mejor or MEJOR
    audio = cargar_audio(ruta_audio)
    pA, mA = _modelo("A")
    usa_C = mejor.startswith("C") or mejor == "D"
    usa_llm = mejor in ("B", "D")
    rA = transcribir_audios([audio], pA, mA, nbest=CFG["nbest"] if (usa_llm and not usa_C) else 0, desc="A")[0]
    if usa_C:
        pC, mC = _modelo("C")
        r = transcribir_audios([audio], pC, mC, nbest=CFG["nbest"] if usa_llm else 0, desc="C")[0]
    else:
        r = rA
    salida = r["hyp"]
    if mejor != "A" and mejor != "C":
        salida = postprocesar_reglas(salida)
    if usa_llm:
        salida = corregir(r["hyp"], r["nbest"], EJEMPLOS)
    print(f"Whisper (A): {rA['hyp']}\n{mejor:>11}: {salida}")
    return {"A": rA["hyp"], mejor: salida}

if len(demo):
    transcribir(demo.iloc[0].ruta)

## 11. (Opcional) App en Gradio
Micrófono o archivo → transcripción de Whisper estándar y del mejor enfoque, lado a lado. `share=True` genera un
link público temporal (72 h) útil para mostrar desde el celular. En Kaggle requiere *Internet on*.

In [ ]:
import gradio as gr

def _app(ruta):
    if not ruta:
        return "", ""
    r = transcribir(ruta)
    return r["A"], r[MEJOR]

with gr.Blocks(title="Whisper adaptado a habla con disartria") as app:
    gr.Markdown(f"### Whisper estándar vs. adaptado ({MEJOR})\nGrabá o subí un audio en español.")
    entrada = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Audio")
    with gr.Row():
        salida_a = gr.Textbox(label="Whisper estándar (A)", lines=6)
        salida_m = gr.Textbox(label=f"Mejor enfoque ({MEJOR})", lines=6)
    gr.Button("Transcribir", variant="primary").click(_app, entrada, [salida_a, salida_m])
    ejemplos = sorted(glob.glob(os.path.join(DEMO_DIR, "*.wav")))
    if ejemplos:
        gr.Examples(ejemplos, entrada)
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    print("Corrida en segundo plano (Save & Run All): no lanzo Gradio. Abrí el notebook en modo interactivo para la demo.")
else:
    app.launch(share=True, debug=False)